In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported and style set")



✅ Libraries imported and style set


In [3]:
DATA_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Dataset\\integrated_dataset.csv"

try:
    df = pd.read_csv(DATA_PATH)
    print(f"\n✓ Loaded dataset: {df.shape}")
except FileNotFoundError:
    print(f"\n❌ Error: File not found at {DATA_PATH}")
    print("Please run the data integration notebook/script first")



✓ Loaded dataset: (606000, 24)


In [4]:
print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nColumn types:")
print(df.dtypes.value_counts())

print("\nFirst 3 rows:")
df.head(3)



Shape: 606000 rows × 24 columns
Memory usage: 445.73 MB

Column types:
float64    11
object      9
int64       4
Name: count, dtype: int64

First 3 rows:


,product_id,product_name,product_category,product_weight_kg,fragility_index,shipping_type,material_id,material_type,packaging_type,suitable_categories,...,co2_emission_per_kg,load_handling_score,moisture_resistance,thermal_resistance,cost_per_unit_usd,supplier_region,reusability_percent,recycled_content_percent,waste_reduction_impact,compatibility_score
0,1,Chocolate Box,Food,0.5,2,Air,MAT_0001,Cardboard,Cardboard Boxes,"E-commerce, Food & Beverage, Consumer Goods, A...",...,0.54,6.0,5.0,4.0,2.24,EMEA,49.0,79.0,61.0,100
1,1,Chocolate Box,Food,0.5,2,Air,MAT_0002,Paper/Bio-Based,Protective Fillers (Paper/Biodegradable),"Fragile Items, Cosmetics, Pharmaceuticals, Int...",...,0.32,3.0,3.0,3.0,1.82,APAC,31.0,93.0,68.0,35
2,1,Chocolate Box,Food,0.5,2,Air,MAT_0003,Steel,Steel Racks & Containers,"Heavy Industrial Components, High-Security Goods",...,3.25,8.0,9.0,9.0,25.00,AMERICAS,100.0,78.0,85.0,60


In [5]:
missing = df.isnull().sum()
missing_percent = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing_Count': missing.values,
    'Missing_Percent': missing_percent.values
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_df) > 0:
    print("\nColumns with missing values:")
    missing_df
else:
    print("\n✓ No missing values found!")



✓ No missing values found!


In [6]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(f"Numerical columns: {len(numeric_cols)}")

print("\nSummary statistics:")
df[numeric_cols].describe().round(2)

# Outlier detection (IQR method)
outlier_summary = {}
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    if len(outliers) > 0:
        outlier_summary[col] = len(outliers)

if outlier_summary:
    print("\nOutliers detected:")
    outlier_summary
else:
    print("\n✓ No significant outliers detected")


Numerical columns: 15

Summary statistics:

Outliers detected:


In [7]:
categorical_cols = df.select_dtypes(include=['object']).columns
print(f"Categorical columns: {len(categorical_cols)}")

for col in categorical_cols:
    unique_count = df[col].nunique()
    print(f"\n{col}: {unique_count} unique values")
    if unique_count <= 10:
        print(df[col].value_counts())
    else:
        print(df[col].value_counts().head(5))


Categorical columns: 9

product_name: 1480 unique values
product_name
LED Monitor 32"               808
Travel Tumbler - Insulated    808
Smartphone - Lite             808
Glass Bottle - Premium        808
Energy Drink - Zero Sugar     808
Name: count, dtype: int64

product_category: 7 unique values
product_category
Food             138572
Electronics      118776
Cosmetics        109888
Paper Product     93324
Pharmacy          84032
Drinkware         59792
 Cosmetics         1616
Name: count, dtype: int64

shipping_type: 3 unique values
shipping_type
Road    374912
Air     187860
Sea      43228
Name: count, dtype: int64

material_id: 404 unique values
material_id
MAT_0404    1500
MAT_0001    1500
MAT_0002    1500
MAT_0003    1500
MAT_0388    1500
Name: count, dtype: int64

material_type: 4 unique values
material_type
Plastic            298500
Cardboard          208500
Paper/Bio-Based     55500
Steel               43500
Name: count, dtype: int64

packaging_type: 6 unique values
packagi

In [8]:
key_numeric_cols = [
    'product_weight_kg', 'fragility_index', 'recyclability_percent',
    'carbon_footprint', 'co2_emission_per_kg', 'cost_per_unit_usd',
    'load_handling_score', 'moisture_resistance', 'thermal_resistance',
    'compatibility_score'
]

available_cols = [col for col in key_numeric_cols if col in df.columns]
corr_matrix = df[available_cols].corr()

# Display top correlations
corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_pairs.append({
            'Feature_1': corr_matrix.columns[i],
            'Feature_2': corr_matrix.columns[j],
            'Correlation': corr_matrix.iloc[i, j]
        })

corr_df = pd.DataFrame(corr_pairs).sort_values('Correlation', ascending=False)
corr_df.head(10)


,Feature_1,Feature_2,Correlation
33,co2_emission_per_kg,thermal_resistance,0.891523
42,moisture_resistance,thermal_resistance,0.887241
30,co2_emission_per_kg,cost_per_unit_usd,0.879871
24,carbon_footprint,co2_emission_per_kg,0.872429
32,co2_emission_per_kg,moisture_resistance,0.868966
39,load_handling_score,moisture_resistance,0.847450
40,load_handling_score,thermal_resistance,0.821555
31,co2_emission_per_kg,load_handling_score,0.774121
37,cost_per_unit_usd,thermal_resistance,0.772242
36,cost_per_unit_usd,moisture_resistance,0.752055


In [9]:
issues = []

# Check duplicates
duplicates = df.duplicated().sum()
if duplicates > 0:
    issues.append(f"Found {duplicates} duplicate rows")

# Check negative values
positive_cols = ['product_weight_kg', 'cost_per_unit_usd', 'carbon_footprint']
for col in positive_cols:
    if col in df.columns:
        negative_count = (df[col] < 0).sum()
        if negative_count > 0:
            issues.append(f"{col} has {negative_count} negative values")

# Check percentage columns outside 0-100
percent_cols = [col for col in df.columns if 'percent' in col.lower()]
for col in percent_cols:
    out_of_range = ((df[col] < 0) | (df[col] > 100)).sum()
    if out_of_range > 0:
        issues.append(f"{col} has {out_of_range} values outside 0-100")

# Check score columns outside 1-10
score_cols = [col for col in df.columns if 'score' in col.lower() and col != 'compatibility_score']
for col in score_cols:
    if col in df.columns:
        out_of_range = ((df[col] < 1) | (df[col] > 10)).sum()
        if out_of_range > 0:
            issues.append(f"{col} has {out_of_range} values outside 1-10")

if issues:
    print("\n⚠ Data quality issues found:")
    for i, issue in enumerate(issues, 1):
        print(f"  {i}. {issue}")
else:
    print("\n✓ No data quality issues detected!")



✓ No data quality issues detected!


In [10]:
report_path = "C:/Users/sneha/Desktop/ecopackai/Data/docs/eda_report.txt"
os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, 'w') as f:
    f.write("EcoPackAI - Exploratory Data Analysis Report\n")
    f.write("="*60 + "\n\n")
    f.write(f"Dataset shape: {df.shape}\n")
    f.write(f"Total records: {len(df)}\n")
    f.write(f"Total features: {len(df.columns)}\n\n")
    
    f.write("Missing Values:\n")
    if len(missing_df) > 0:
        f.write(missing_df.to_string(index=False) + "\n\n")
    else:
        f.write("No missing values\n\n")
    
    f.write("Data Quality Issues:\n")
    if issues:
        for issue in issues:
            f.write(f"  - {issue}\n")
    else:
        f.write("  No issues found\n")

print(f"✓ EDA report saved to: {report_path}")

# Optional: Save missing values table
if len(missing_df) > 0:
    missing_df.to_csv(os.path.join(os.path.dirname(report_path), "missing_values_table.csv"), index=False)
    print("✓ Missing values table saved")


✓ EDA report saved to: C:/Users/sneha/Desktop/ecopackai/Data/docs/eda_report.txt
